In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook


In [6]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)

2 + 2 = 4


In [33]:
# Create a quick script to list models
models = gemini.models.list()
for model in models.data:
    print(model.id)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [13]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, Gemini! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, Gemini! This is my first ever message to you! Hi!'}]

In [14]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=messages)
response.choices[0].message.content

"Hello there! It's fantastic to meet you! A very warm welcome to our first conversation. I'm delighted you reached out.\n\nHow can I help you today, or what's on your mind?"

In [16]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 500,000 enrolled across 194

In [17]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [18]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [19]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=messages)
response.choices[0].message.content

'2 + 2 = 4'

In [20]:
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=messages)
response.choices[0].message.content

"Oh, the profound questions you bring. After extensive calculations and consulting ancient texts, I can confidently tell you it's **4**. You're welcome."

## And now let's build useful messages for Gemini-2.5-flash, using a function

In [21]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

## Time to bring it together - the API for OpenAI is very simple!

In [23]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = gemini.chat.completions.create(
        model = "gemini-2.5-flash",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [24]:
summarize("https://edwarddonner.com")

'This website belongs to Edward Donner, a man who, if left unsupervised, will drone on about LLMs until your ears bleed, which clearly paid off given his *best-selling, top-rated* Udemy courses. He\'s also busy applying AI to help you "discover your potential," which sounds suspiciously like he wants to help LLMs take over the world, one self-discovery journey at a time.\n\n**News and Announcements (from the future, apparently):**\nLooks like Ed\'s got a crystal ball because his blog posts are dated 2025 and 2026, offering resources for AI coders and builders, MLOps tracks, and, most importantly, a guide on "Which order to take the AI courses?" – because you *will* be taking them.'

In [25]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [26]:
display_summary("https://edwarddonner.com")

This website is the digital playground of Edward Donner, a man so enthusiastic about LLMs that his friends paid him to take it elsewhere – specifically, to a best-selling Udemy curriculum. When he's not teaching 500,000 students or helping people find their purpose with AI (at Nebula.io), he's busy building arenas where LLMs can battle it out with "diplomacy and deviousness."

As for news, expect more deep dives into his AI courses, with new resources for "AI Coder: Vibe Coder to Agentic Engineer" (Feb 2026), "AI Builder with n8n" (Jan 2026), and an "AI Engineering MLOps Track" (Sept 2025), because apparently, you can never have enough AI education. There's also guidance on which order to take his courses (May 2025) – a real brain-teaser, I'm sure.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [27]:
display_summary("https://cnn.com")

This website is a glorious ad-feedback form, masquerading as a news site. It's truly dedicated to letting you complain about all the ways their ads break your experience, while also offering an exhaustive list of every single topic ever conceived by humanity – just in case you ever get past the commercials to find actual content.

In [28]:
display_summary("https://anthropic.com")

This website is for Anthropic, an AI company earnestly dedicated to making sure AI doesn't doom us all, while conveniently selling their AI assistant, Claude, in multiple flavors (Code, Cowork, Platform – oh my!). They've also got new models named after classical music.

As for news, they're securing critical software with 'Project Glasswing', just released a "smarter, more capable" Claude Opus 4.7 (due April 16, 2026), and apparently, Claude is a serene "space to think" free of pesky ads (as of February 4, 2026). Imagine that!

In [30]:
display_summary("https://portfolio-sm-iota.vercel.app/")

This website is a stunningly minimalist masterpiece, featuring only its own name. Truly, a less-is-more approach to online presence that leaves you wondering, "Is that all there is, or did someone forget to paste the rest of the internet?"

In [36]:
# Step 1: Create your prompts

system_prompt = "You are a patient computer science tutor who explains concepts using simple language, analogies, and step-by-step breakdowns."
user_prompt = "Explain hash tables like I am in class 10."

# Step 2: Make the messages list

messages = [
   {"role": "system", "content" : system_prompt},
   {"role": "user", "content" : user_prompt}
] # fill this in

# Step 3: Call Gemini
response = gemini.chat.completions.create(
        model = "gemini-2.5-flash",
        messages = messages
    )

# Step 4: print the result
print(response.choices[0].message.content)

Alright, imagine you have a big pile of things you need to keep track of – maybe it's all your school notes, or all the books in a library, or even all the different games on your computer.

The big question is: **How do you find exactly what you're looking for, super fast, without having to search through everything every single time?**

Let's break it down!

---

### The Problem: Finding Stuff Quickly

1.  **The "Slow" Way:**
    Imagine you have a big stack of flashcards with vocabulary words. If they're just in a random pile, and you want to find "photosynthesis," what do you do? You start from the top and flip through them, one by one, until you find it. This works, but it can be really slow, especially if you have thousands of flashcards!

2.  **The "Better" Way (if possible):**
    What if you could organize them? Like, if you could put all the "A" words in one box, "B" words in another, and so on. Or, if they were numbers, you could put number 1 in spot 1, number 2 in spot 2, e

In [ ]:
import asyncio
from playwright.async_api import async_playwright

# Fix for Windows + Jupyter
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

async def scrape_website(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        await page.goto(url)
        await page.wait_for_timeout(3000)
        
        content = await page.content()
        await browser.close()
        
        return content

html = await scrape_website("https://portfolio-sm-iota.vercel.app/")
print(html[:1000])

In [ ]:
from bs4 import BeautifulSoup

def extract_text(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text()


text = extract_text(html)
print(text[:1000])

In [ ]:
prompt = f"""
You are a patient critic.

Explain the following content in simple terms:

{text[:3000]}
"""

response = model.generate_content(prompt)
print(response.text)